# MicroPathMamba: training on a Kaggle GPU

**Before running**
1. Click **Settings** in the right panel.
   - **Accelerator:** choose **GPU T4 x2**. The MambaVision kernels need a T4 or newer, so don't pick P100.
   - **Internet:** turn **On**. This needs a phone-verified Kaggle account.
2. Edit `RUNS` in the next cell. The default is Phase 1: fold 0, seed 1. Each session lasts up to 12 hours.
3. Click **Save Version**, then **Save & Run All (Commit)**. The notebook keeps running after you close the tab.
4. When it finishes, open the version, go to **Output**, and download `kaggle_results.zip`.
5. Unzip it into `~/PycharmProjects/MicroPathMamba-STM32N6/` on the Mac, so the `runs/` folder merges. Then run `bash scripts/mac_run.sh`.

Runs that already have a `metrics.json` are skipped. To continue in a later session, upload the previous `kaggle_results.zip` as a Kaggle dataset and set `RESUME_FROM`.

In [ ]:
# ---- configuration -------------------------------------------------------------
REPO = "https://github.com/kishorravi/MicroPathMamba-STM32N6.git"
EPOCHS = 30
# (model, fold, seed, teacher or None). Teachers must come before the students that use them.
RUNS = [
    ("mambavision_t",      0, 1, None),
    ("micropathmamba",     0, 1, "mambavision_t"),   # distilled student (main result)
    ("micropathmamba",     0, 1, None),              # student without distillation
    ("micropathmamba_cnn", 0, 1, None),              # ablation: no SSM blocks
    ("mobilenetv3_s",      0, 1, None),              # MCU baseline
    ("convnextv2_t",       0, 1, None),              # strong CNN baseline
]
RESUME_FROM = None   # e.g. "/kaggle/input/micropathmamba-results/kaggle_results.zip"

In [ ]:
import os, subprocess, sys, shutil, time, json, glob
W = "/kaggle/working"; CODE = f"{W}/MicroPathMamba-STM32N6"; DATA = "/tmp/data"
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode: raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r.returncode
if os.path.isdir(CODE): sh(f"cd {CODE} && git pull --ff-only")
else: sh(f"git clone --depth 1 {REPO} {CODE}")
os.chdir(CODE)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("pip install -q timm onnx onnxruntime fvcore einops")

In [ ]:
# MambaVision needs mamba-ssm. Try the official CUDA package (prebuilt wheel or build) for up to 40 min;
# if that fails, fall back to the pure-PyTorch scan in kaggle/mamba_ssm_shim (same maths, slower).
SHIM = False
try:
    rc = subprocess.run("pip install -q causal-conv1d mamba-ssm --no-build-isolation", shell=True, timeout=2400).returncode
    import importlib; import mamba_ssm; importlib.import_module("mamba_ssm.ops.selective_scan_interface")
    print("official mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("mamba-ssm unavailable -> using pure-PyTorch shim:", repr(e)[:200])
    sh(f"pip install -q -e {CODE}/kaggle/mamba_ssm_shim || true", check=False)
    sys.path.insert(0, f"{CODE}/kaggle/mamba_ssm_shim")
    os.environ["PYTHONPATH"] = f"{CODE}/kaggle/mamba_ssm_shim:" + os.environ.get("PYTHONPATH", "")
    SHIM = True
sh("pip install -q --no-deps mambavision")
sh(f'python -c "from models import build_model; m=build_model(\'mambavision_t\', 9, pretrained=False); print(\'MambaVision OK\', sum(p.numel() for p in m.parameters())/1e6, \'M\')"')
json.dump({"mamba_ssm_shim": SHIM}, open(f"{W}/environment.json", "w"))

In [ ]:
# ---- data (about 12 GB, kept in /tmp so /kaggle/working stays small) ----------
os.makedirs(DATA, exist_ok=True)
for name in ["CRC-VAL-HE-7K", "NCT-CRC-HE-100K"]:
    if not os.path.isdir(f"{DATA}/{name}"):
        sh(f'curl -sSL -o {DATA}/{name}.zip "https://zenodo.org/records/1214456/files/{name}.zip?download=1"')
        sh(f"cd {DATA} && unzip -q -o {name}.zip && rm {name}.zip")
    print(name, sum(len(f) for _, _, f in os.walk(f"{DATA}/{name}")), "images")
sh(f"python scripts/make_folds.py --data {DATA}/NCT-CRC-HE-100K --out folds.csv --seed 123")
if RESUME_FROM: sh(f"cd {W} && unzip -q -o {RESUME_FROM}")

In [ ]:
# ---- training ------------------------------------------------------------------
OUT = f"{W}/runs"
def tag(m, t): return m + (f"_kd-{t}" if t else "")
for m, f, s, t in RUNS:
    done = f"{OUT}/{tag(m, t)}/f{f}_s{s}/metrics.json"
    if os.path.exists(done): print("skip", tag(m, t), f, s); continue
    cmd = (f"python train.py --model {m} --fold {f} --seed {s} --epochs {EPOCHS} --workers 4 "
           f"--data {DATA}/NCT-CRC-HE-100K --ext {DATA}/CRC-VAL-HE-7K --out {OUT}")
    if t: cmd += f" --teacher {t} --teacher-ckpt {OUT}/{t}/f{f}_s{s}/best.pt"
    t0 = time.time(); rc = sh(cmd + f" 2>&1 | tee {W}/log_{tag(m, t)}_f{f}_s{s}.txt", check=False)
    print(f"--- {tag(m, t)} f{f} s{s} finished in {(time.time()-t0)/3600:.2f} h (rc={rc})")

In [ ]:
# ---- package results for the Mac -----------------------------------------------
sh(f"cd {CODE} && python scripts/complexity.py --models micropathmamba micropathmamba_cnn mobilenetv3_s convnextv2_t mambavision_t --out {W}/results/complexity_gpu.csv", check=False)
sh(f"cd {W} && rm -f kaggle_results.zip && zip -qr kaggle_results.zip runs results environment.json log_*.txt")
for p in sorted(glob.glob(f"{W}/runs/*/f*_s*/metrics.json")):
    r = json.load(open(p)); e = r.get("external_7k", {})
    print(f"{p.split('/runs/')[1]:55s} CV acc {r['internal_cv']['accuracy']:.4f} | 7K acc {e.get('accuracy', float('nan')):.4f} | 7K F1 {e.get('f1_macro', float('nan')):.4f}")
print("Download kaggle_results.zip from the Output tab.")